# FrenzAI - Cloud TTS Studio
Run FrenzAI with GPU acceleration on Google Colab.

**Instructions:**
1. Make sure GPU runtime is enabled: `Runtime > Change runtime type > T4 GPU`
2. Run all cells in order (click the play button on each)
3. Click the public URL at the end to open FrenzAI in your browser

In [ ]:
#@title 1. Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
#@title 2. Clone FrenzAI & Install Dependencies (takes ~5 min)
import os

# Clone the repo
if not os.path.exists('/content/FrenzAI'):
    !git clone https://github.com/YOUR_USERNAME/FrenzAI.git /content/FrenzAI
else:
    !cd /content/FrenzAI && git pull

os.chdir('/content/FrenzAI')

# Install system deps
!apt-get update -qq && apt-get install -y -qq ffmpeg sox espeak-ng nodejs npm > /dev/null 2>&1

# Install Python deps
!pip install -q -r backend/requirements.txt
!pip install -q kokoro kokoro-onnx edge-tts espeakng-loader
!pip install -q chatterbox-tts resemble-perth
!pip install -q --no-deps f5-tts
!pip install -q accelerate safetensors transformers vocos torchdiffeq einops x_transformers cached_path pypinyin rjieba unidecode tomli librosa hydra-core omegaconf ema_pytorch
!pip install -q --no-deps qwen-tts
!pip install -q onnxruntime
!pip install -q --no-deps parler-tts descript-audio-codec descript-audiotools
!pip install -q flatten-dict julius argbind tensorboard ffmpy
!pip install -q --no-deps fish-speech
!pip install -q pyloudnorm pedalboard noisereduce

# Build frontend
!cd frontend && npm install --silent 2>/dev/null && npm run build

# Create data dirs
!mkdir -p data/voices/samples data/voices/embeddings data/voices/import data/voices/staging
!mkdir -p data/projects data/exports data/temp backend/models

print('\n--- DONE! All dependencies installed ---')

In [ ]:
#@title 3. Upload Voice Files (Optional)
#@markdown Upload your voice samples (.opus, .wav, .mp3) for cloning.
#@markdown They'll be placed in the import folder.

from google.colab import files
import shutil, os

os.chdir('/content/FrenzAI')
IMPORT_DIR = 'data/voices/import'
os.makedirs(IMPORT_DIR, exist_ok=True)

print('Select voice files to upload (or skip this cell)...')
try:
    uploaded = files.upload()
    for fname, content in uploaded.items():
        dest = os.path.join(IMPORT_DIR, fname)
        with open(dest, 'wb') as f:
            f.write(content)
        print(f'  Uploaded: {fname} ({len(content)/1024:.0f} KB)')
    print(f'\nTotal: {len(uploaded)} files uploaded to {IMPORT_DIR}/')
except Exception as e:
    print(f'Upload skipped or cancelled: {e}')

In [ ]:
#@title 4. Start FrenzAI Server
import subprocess, time, os, threading

os.chdir('/content/FrenzAI/backend')
os.environ['FRENZAI_CLOUD'] = '1'
os.environ['VOICEFORGE_HOST'] = '0.0.0.0'
os.environ['VOICEFORGE_PORT'] = '8111'
os.environ['VOICEFORGE_DEVICE'] = 'cuda'

# Start backend in background
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8111'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

# Stream logs in background
def stream_logs():
    for line in iter(proc.stdout.readline, b''):
        print(line.decode().strip())

log_thread = threading.Thread(target=stream_logs, daemon=True)
log_thread.start()

# Wait for server to be ready
import urllib.request
for i in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8111/api/health', timeout=2)
        print('\n--- FrenzAI backend is READY! ---')
        break
    except:
        time.sleep(2)
        if i % 5 == 0:
            print(f'  Waiting for server... ({i*2}s)')
else:
    print('ERROR: Server did not start in 120 seconds')

In [ ]:
#@title 5. Create Public URL (click this link to open FrenzAI!)
from google.colab import output

# Method 1: Try pyngrok for a clean public URL
try:
    !pip install -q pyngrok
    from pyngrok import ngrok
    
    # Kill any existing tunnels
    ngrok.kill()
    
    # Create tunnel
    tunnel = ngrok.connect(8111)
    public_url = tunnel.public_url
    print(f'\n{"="*50}')
    print(f'  FrenzAI is LIVE!')
    print(f'  Open this URL in your browser:')
    print(f'  {public_url}')
    print(f'{"="*50}')
except Exception as e:
    print(f'ngrok failed ({e}), using Colab proxy instead...')
    # Method 2: Colab's built-in port forwarding
    from google.colab.output import eval_js
    public_url = eval_js("google.colab.kernel.proxyPort(8111)")
    print(f'\n{"="*50}')
    print(f'  FrenzAI is LIVE!')
    print(f'  Open this URL:')
    print(f'  {public_url}')
    print(f'{"="*50}')

In [ ]:
#@title 6. Download Generated Audiobooks (run when done)
#@markdown Run this cell to download your generated audio files before the session ends.

from google.colab import files
import os, zipfile

os.chdir('/content/FrenzAI')

export_dir = 'data/exports'
if os.path.exists(export_dir) and os.listdir(export_dir):
    # Zip all exports
    zip_path = '/content/frenzai_exports.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in os.listdir(export_dir):
            fp = os.path.join(export_dir, f)
            zf.write(fp, f)
            print(f'  Added: {f}')
    
    print(f'\nDownloading {zip_path}...')
    files.download(zip_path)
else:
    print('No exports found. Generate some audiobooks first!')